In [2]:
import duckdb

duckdb.sql("""
           CREATE OR REPLACE TABLE filtered AS
           SELECT message_id, channel_id, user_id, date, text_content, reply_to, n_forwards,
           views, reactions, forward_from, is_vaccine_related, language
           FROM read_json_auto('telegram.jsonl')
           WHERE text_content NOT NULL AND text_content != ''
           AND (language = 'Portuguese' OR language = 'English' OR language = 'Spanish')
           """)

# transform timestamp utc from mili to sec
duckdb.sql("UPDATE filtered SET date = date/1000")

In [42]:
duckdb.sql("""
           SELECT language, COUNT(*) FROM filtered
           GROUP BY language
           """).show()

┌────────────┬──────────────┐
│  language  │ count_star() │
│  varchar   │    int64     │
├────────────┼──────────────┤
│ Portuguese │      2331403 │
│ English    │       320762 │
│ Spanish    │        71014 │
└────────────┴──────────────┘



# Channel table

In [ ]:
# the reply_to field only contains references to messages within the same channel.
# because of that, only the forward_from field was considered. 
duckdb.sql("""
           CREATE OR REPLACE VIEW channels AS
           SELECT channel_id, forward_from
           FROM filtered
           WHERE forward_from NOT NULL
           AND forward_from NOT LIKE '<USER%'
           AND channel_id != forward_from
           """)

duckdb.sql("SELECT COUNT(*) FROM channels")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       337657 │
└──────────────┘

In [44]:
# Limit the forward_from references to just the 119 channels that had their messages
# tracked on the original dataset.
# Besides, group in pairs the message channel and the original channel (channel_id and
# forward_from) to compose the future graph edges.

duckdb.sql("""
           CREATE OR REPLACE VIEW channels2 AS
           SELECT channel_id, forward_from, count(*) AS weight
           FROM channels
           WHERE forward_from IN (
                SELECT DISTINCT channel_id
                FROM channels)
           GROUP BY channel_id, forward_from
           """)
duckdb.sql("SELECT COUNT(*) FROM channels2")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1731 │
└──────────────┘

In [45]:
duckdb.sql("COPY channels2 TO 'channels.csv' (HEADER, DELIMITER ',')")

# Chat table

In [58]:
# as the forward_from field only specifies the channel from which
# the message came, it is necessary to search for the orginal 
# forwarded message and link them.

duckdb.sql("""
           CREATE OR REPLACE VIEW fwd_messages AS
           SELECT message_id, channel_id, text_content, forward_from
           FROM filtered
           WHERE forward_from NOT NULL
           AND forward_from != ''
           AND forward_from IN (
               SELECT DISTINCT channel_id
               FROM filtered
               )
           """)

# original message identified as two messages with same text content
# and one of them referencing the channel of the other.
duckdb.sql("""
           CREATE OR REPLACE TABLE new_fwd_messages AS
           SELECT a.message_id AS message_id, a.text_content AS text_content,
           b.message_id AS new_forward_from
           FROM fwd_messages a JOIN filtered b
           ON a.text_content = b.text_content
           AND a.forward_from = b.channel_id
           AND a.message_id != b.message_id
           """)

In [51]:
duckdb.sql("""COPY(
           SELECT message_id, count(*)
           FROM new_fwd_messages
           GROUP BY message_id, text_content
           ORDER BY count(*) DESC)
           TO 'temp.csv' (HEADER, DELIMITER ',')
           """)

In [60]:
# after manual inspection, it was seen that all messages that had high
# duplicity rates and were above 93 matches had their content only about
# channel link sharing or channel rules. It was decided to remove those
# links from analysis.
duckdb.sql("""
           DELETE FROM new_fwd_messages
           WHERE message_id IN (
                SELECT message_id
                FROM new_fwd_messages
                GROUP BY message_id, text_content
                HAVING COUNT(*) > 92
           )
           """)

print("Forwarded messages:")
print(duckdb.sql("SELECT COUNT(*) FROM fwd_messages"))
print("Total pairs origin-message:")
print(duckdb.sql("SELECT COUNT(*) FROM new_fwd_messages"))
print("Unique pairs origin-message:")
print(duckdb.sql("SELECT COUNT(distinct message_id) FROM new_fwd_messages"))

Forwarded messages:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       225704 │
└──────────────┘

Total pairs origin-message:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       307156 │
└──────────────┘

Unique pairs origin-message:
┌────────────────────────────┐
│ count(DISTINCT message_id) │
│           int64            │
├────────────────────────────┤
│                     210911 │
└────────────────────────────┘



In [66]:
duckdb.sql("""
           CREATE OR REPLACE TABLE fwd_insert AS
           SELECT a.message_id AS message_id, a.channel_id AS channel_id,
           a.user_id AS user_id, a.date AS date, a.text_content AS text_content,
           a.reply_to AS reply_to, a.n_forwards AS n_forwards, a.views AS views,
           a.reactions AS reactions, b.new_forward_from AS forward_from,
           a.is_vaccine_related AS is_vaccine_related, a.language AS language
           FROM filtered AS a, new_fwd_messages AS b
           WHERE a.message_id = b.message_id
           """)
duckdb.sql("SELECT count(*) FROM fwd_insert")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       307156 │
└──────────────┘

In [67]:
# Updating the forwarding links to the filtered table by
# removing old messages and inserting the new ones.
# (this may introduce repeated messages that have the same
# info and id, but each one will have a different link and
# that will be used to construct the graph edges)
duckdb.sql("""
           DELETE FROM filtered
           WHERE message_id IN (
                SELECT DISTINCT message_id
                FROM fwd_insert)
           """)
duckdb.sql("""
           INSERT INTO filtered
           SELECT *
           FROM fwd_insert
           """)
